## Gai Dialogue System

This system is responsible for managing and storing the dialogue messages between the user and the agents. It also handles the context and history of the conversation. It has a dependency on `Gai Messages System`.


---

## 1. Dialogue Bus

### a) Setup

In [ ]:
from gai.dialogue import DialogueBus
from gai.dialogue.nodes import AgentNode, UserNode
from rich import print as rprint

class MockAgentNode(AgentNode):
    
    def __init__(self, name: str, dialogue: DialogueBus):
        super().__init__(name=name, dialogue=dialogue, handle_send_cb=self.handle_send_cb)
        
    def handle_send_cb(self, message):
        
        # Simulate a response from the agent
        
        if self.name == "Sara":
            message = "This is a story about a brave knight who fought a dragon.".split()
            for word in message:
                yield word+" "
        if self.name == "Diana":
            message = "The knight was very brave and fought valiantly.".split()
            for word in message:
                yield word+" "

# 1. Initiate a dialogue session

dialogue = DialogueBus()
await dialogue.start()

# 2. Add 1 User and 2 Agents to the Dialogue
    
sara = MockAgentNode(name="Sara", dialogue=dialogue)
diana = MockAgentNode(name="Diana", dialogue=dialogue)

async def handle_reply_cb(message):
    rprint(f"[bold green]{message.header.sender} replied:[/bold green] {message.body.content}")

user = UserNode(dialogue=dialogue, handle_reply_cb=handle_reply_cb)


/workspaces/gai-sdk/src/gai/dialogue/nodes/user_node.py:46: RuntimeWarning: coroutine 'handle_reply_cb' was never awaited
  self.handle_reply_cb(message)


### b) Send and Reply

In [2]:
# 3. Ask Sara to tell a story

await user.chat("Tell me a one paragraph story","Sara") 
rprint("[User] [bold bright_yellow] Sent message to Sara [/bold bright_yellow]")

# 4. Ask Diana to continue   

await user.chat("Please continue","Diana")
rprint("[User] [bold bright_yellow] Sent message to Diana [/bold bright_yellow]")



[User]  Sent message to Sara 

[User]  Sent message to Diana 

[Sara] Handle send message. message=id='cae49246-6879-4998-aa8a-925456693292' 
header=MessageHeaderPydantic(sender='User', recipient='Sara', timestamp=1752027451.7352684, order=0) 
body=SendBodyPydantic(type='send', dialogue_id='00000000-0000-0000-0000-000000000000', message_no=1, 
message_id='00000000-0000-0000-0000-000000000000.1', content_type='text', content='Tell me a one paragraph story')

[Diana] Message not for me, ignore. message=id='cae49246-6879-4998-aa8a-925456693292' 
header=MessageHeaderPydantic(sender='User', recipient='Sara', timestamp=1752027451.7352684, order=0) 
body=SendBodyPydantic(type='send', dialogue_id='00000000-0000-0000-0000-000000000000', message_no=1, 
message_id='00000000-0000-0000-0000-000000000000.1', content_type='text', content='Tell me a one paragraph story')

[Sara] Message not for me, ignore. message=id='4c06cd10-c67b-4322-a9fa-1dcedd52062e' 
header=MessageHeaderPydantic(sender='User', recipient='Diana', timestamp=1752027451.744458, order=0) 
body=SendBodyPydantic(type='send', dialogue_id='00000000-0000-0000-0000-000000000000', message_no=2, 
message_id='00000000-0000-0000-0000-000000000000.2', content_type='text', content='Please continue')

[Diana] Handle send message. message=id='4c06cd10-c67b-4322-a9fa-1dcedd52062e' 
header=MessageHeaderPydantic(sender='User', recipient='Diana', timestamp=1752027451.744458, order=0) 
body=SendBodyPydantic(type='send', dialogue_id='00000000-0000-0000-0000-000000000000', message_no=2, 
message_id='00000000-0000-0000-0000-000000000000.2', content_type='text', content='Please continue')

### c) Send broadcast messages

In [3]:
# 1. Send a broadcast message to all agents

await user.chat("Introduce yourselves","*") 
print("[User] [bold bright_yellow] Sent broadcast message [/bold bright_yellow]")



[User] [bold bright_yellow] Sent broadcast message [/bold bright_yellow]


[Sara] Handle broadcast message. message=id='77361fb5-5b0f-4025-ab22-3ff6a8c04b0b' 
header=MessageHeaderPydantic(sender='User', recipient='*', timestamp=1752027459.6155035, order=0) 
body=SendBodyPydantic(type='send', dialogue_id='00000000-0000-0000-0000-000000000000', message_no=25, 
message_id='00000000-0000-0000-0000-000000000000.25', content_type='text', content='Introduce yourselves')

[Diana] Handle broadcast message. message=id='77361fb5-5b0f-4025-ab22-3ff6a8c04b0b' 
header=MessageHeaderPydantic(sender='User', recipient='*', timestamp=1752027459.6155035, order=0) 
body=SendBodyPydantic(type='send', dialogue_id='00000000-0000-0000-0000-000000000000', message_no=25, 
message_id='00000000-0000-0000-0000-000000000000.25', content_type='text', content='Introduce yourselves')

In [4]:
print(dialogue.list_messages())

[MessagePydantic(id='cae49246-6879-4998-aa8a-925456693292', header=MessageHeaderPydantic(sender='User', recipient='Sara', timestamp=1752027451.7352684, order=0), body=SendBodyPydantic(type='send', dialogue_id='00000000-0000-0000-0000-000000000000', message_no=1, message_id='00000000-0000-0000-0000-000000000000.1', content_type='text', content='Tell me a one paragraph story')), MessagePydantic(id='4c06cd10-c67b-4322-a9fa-1dcedd52062e', header=MessageHeaderPydantic(sender='User', recipient='Diana', timestamp=1752027451.744458, order=0), body=SendBodyPydantic(type='send', dialogue_id='00000000-0000-0000-0000-000000000000', message_no=2, message_id='00000000-0000-0000-0000-000000000000.2', content_type='text', content='Please continue')), MessagePydantic(id='75814429-2ccf-4c35-a137-60896b4294c9', header=MessageHeaderPydantic(sender='Sara', recipient='User', timestamp=1752027451.7583008, order=0), body=ReplyBodyPydantic(type='reply', dialogue_id='00000000-0000-0000-0000-000000000000', messa

---

## 4. File Dialogue Bus

In [1]:
import asyncio
from gai.messages import message_helper
from gai.dialogue import FileDialogueBus, DialogueBus
from rich import print

from gai.dialogue.nodes import AgentNode, UserNode

class MockAgentNode(AgentNode):
    
    def __init__(self, name: str, dialogue: DialogueBus):
        super().__init__(name=name, dialogue=dialogue)
        
    def _stream_completion(self, message):
        
        # Simulate a response from the agent
        
        if self.name == "Sara":
            yield "This is a story about a brave knight who fought a dragon."
        if self.name == "Diana":
            yield "The knight was very brave and fought valiantly."

# 1. Initiate a dialogue session
from gai.lib.tests import make_local_tmp
here = make_local_tmp()
dialogue_id = FileDialogueBus.create_dialogue_id()

# 2. Add 1 User and 2 Agents to the Dialogue

dialogue = FileDialogueBus(
    logger_name="User", 
    app_dir=here,
    dialogue_id=dialogue_id,
    reset=True
    )
await dialogue.start()
    
sara = MockAgentNode(name="Sara", dialogue=dialogue)
diana = MockAgentNode(name="Diana", dialogue=dialogue)
user = UserNode(dialogue=dialogue)


### b) Send and Reply

In [2]:
# 3. Ask Sara to tell a story

await user.chat("Tell me a one paragraph story","Sara") 
print("[User] [bold bright_yellow] Sent message to Sara [/bold bright_yellow]")

# # 4. Ask Diana to continue   

await user.chat("Please continue","Diana")
print("[User] [bold bright_yellow] Sent message to Diana [/bold bright_yellow]")



[User]  Sent message to Sara 

[User]  Sent message to Diana 

[Sara] Handle send message. message=id='fef4f8e4-2011-4fce-a1c2-bfed6f9c2f1b' 
header=MessageHeaderPydantic(sender='User', recipient='Sara', timestamp=1752039364.021736, order=0) 
body=SendBodyPydantic(type='send', dialogue_id='00000000-0000-0000-0000-000000000000', message_no=1, 
message_id='00000000-0000-0000-0000-000000000000.1', content_type='text', content='Tell me a one paragraph story')

[Diana] Message not for me, ignore. message=id='fef4f8e4-2011-4fce-a1c2-bfed6f9c2f1b' 
header=MessageHeaderPydantic(sender='User', recipient='Sara', timestamp=1752039364.021736, order=0) 
body=SendBodyPydantic(type='send', dialogue_id='00000000-0000-0000-0000-000000000000', message_no=1, 
message_id='00000000-0000-0000-0000-000000000000.1', content_type='text', content='Tell me a one paragraph story')

[Sara] Message not for me, ignore. message=id='822d8d0f-6b49-4fa5-bfd0-38ac5cf7f99a' 
header=MessageHeaderPydantic(sender='User', recipient='Diana', timestamp=1752039364.0299973, order=0) 
body=SendBodyPydantic(type='send', dialogue_id='00000000-0000-0000-0000-000000000000', message_no=2, 
message_id='00000000-0000-0000-0000-000000000000.2', content_type='text', content='Please continue')

[Diana] Handle send message. message=id='822d8d0f-6b49-4fa5-bfd0-38ac5cf7f99a' 
header=MessageHeaderPydantic(sender='User', recipient='Diana', timestamp=1752039364.0299973, order=0) 
body=SendBodyPydantic(type='send', dialogue_id='00000000-0000-0000-0000-000000000000', message_no=2, 
message_id='00000000-0000-0000-0000-000000000000.2', content_type='text', content='Please continue')

### c) Send broadcast messages

In [3]:
# 1. Send a broadcast message to all agents

await user.chat("Introduce yourselves","*") 
print("[User] [bold bright_yellow] Sent broadcast message [/bold bright_yellow]")



[User]  Sent broadcast message 

[Sara] Handle broadcast message. message=id='5bd4d714-fa0d-459a-8c09-999a723b5c6e' 
header=MessageHeaderPydantic(sender='User', recipient='*', timestamp=1752039377.8589463, order=0) 
body=SendBodyPydantic(type='send', dialogue_id='00000000-0000-0000-0000-000000000000', message_no=4, 
message_id='00000000-0000-0000-0000-000000000000.4', content_type='text', content='Introduce yourselves')

[Diana] Handle broadcast message. message=id='5bd4d714-fa0d-459a-8c09-999a723b5c6e' 
header=MessageHeaderPydantic(sender='User', recipient='*', timestamp=1752039377.8589463, order=0) 
body=SendBodyPydantic(type='send', dialogue_id='00000000-0000-0000-0000-000000000000', message_no=4, 
message_id='00000000-0000-0000-0000-000000000000.4', content_type='text', content='Introduce yourselves')

### d) Check File

In [4]:
from gai.lib.constants import DEFAULT_GUID
path = f"tmp/data/{DEFAULT_GUID}/user/dialogue/{dialogue_id}.json"
with open(path,"r") as f:
    text = f.read()
print(text)
    

{
    "last_message_order": 3,
    "messages": [
        {
            "id": "fef4f8e4-2011-4fce-a1c2-bfed6f9c2f1b",
            "header": {
                "sender": "User",
                "recipient": "Sara",
                "timestamp": 1752039364.021736,
                "order": 0
            },
            "body": {
                "type": "send",
                "dialogue_id": "00000000-0000-0000-0000-000000000000",
                "message_no": 5,
                "message_id": "00000000-0000-0000-0000-000000000000.5",
                "content_type": "text",
                "content": "Tell me a one paragraph story"
            }
        },
        {
            "id": "822d8d0f-6b49-4fa5-bfd0-38ac5cf7f99a",
            "header": {
                "sender": "User",
                "recipient": "Diana",
                "timestamp": 1752039364.0299973,
                "order": 0
            },
            "body": {
                "type": "send",
                "dialogue_id": "00000000-0000-0000-0000-000000000000",
                "message_no": 6,
                "message_id": "00000000-0000-0000-0000-000000000000.6",
                "content_type": "text",
                "content": "Please continue"
            }
        },
        {
            "id": "5bd4d714-fa0d-459a-8c09-999a723b5c6e",
            "header": {
                "sender": "User",
                "recipient": "*",
                "timestamp": 1752039377.8589463,
                "order": 0
            },
            "body": {
                "type": "send",
                "dialogue_id": "00000000-0000-0000-0000-000000000000",
                "message_no": 4,
                "message_id": "00000000-0000-0000-0000-000000000000.4",
                "content_type": "text",
                "content": "Introduce yourselves"
            }
        }
    ]
}